In [ ]:
# !pip install yfinance pandas numpy matplotlib scipy ta-lib vectorbt

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import talib
import vectorbt as vbt
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os, sys

In [ ]:
# ============================================================
# EMA HIGH/LOW CHANNEL STRATEGY — Configuration
# ============================================================
# Your personal strategy: 44 EMA(High) / 44 EMA(Low) channel + 200 EMA filter
# Bullish: price above all three → long
# Bearish: price below all three → exit/short

TICKER = 'USDJPY=X'        # Your main pair (change to any asset)
START_DATE = '2018-01-01'
TRAIN_RATIO = 0.60

# EMA parameters (will optimize around these)
CHANNEL_PERIOD = 44         # EMA period for High/Low channel
TREND_PERIOD = 200          # Trend filter EMA

# Backtest settings
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005

print(f"Ticker: {TICKER}")
print(f"Channel EMA: {CHANNEL_PERIOD} | Trend EMA: {TREND_PERIOD}")
print(f"Start: {START_DATE} | Train ratio: {TRAIN_RATIO}")
print(f"Capital: ${INIT_CASH:,} | Fees: {FEES:.2%} | Slippage: {SLIPPAGE:.2%}")

In [ ]:
# ============================================================
# Download Data
# ============================================================
df = yf.download(TICKER, start=START_DATE, auto_adjust=True)
close = df['Close'].squeeze()
high = df['High'].squeeze()
low = df['Low'].squeeze()

split_idx = int(len(close) * TRAIN_RATIO)
train_close, val_close = close.iloc[:split_idx], close.iloc[split_idx:]
train_high, val_high = high.iloc[:split_idx], high.iloc[split_idx:]
train_low, val_low = low.iloc[:split_idx], low.iloc[split_idx:]

print(f"Total bars: {len(close)} ({close.index[0].strftime('%Y-%m-%d')} \u2192 {close.index[-1].strftime('%Y-%m-%d')})")
print(f"Train: {len(train_close)} bars ({train_close.index[0].strftime('%Y-%m-%d')} \u2192 {train_close.index[-1].strftime('%Y-%m-%d')})")
print(f"Test:  {len(val_close)} bars ({val_close.index[0].strftime('%Y-%m-%d')} \u2192 {val_close.index[-1].strftime('%Y-%m-%d')})")
print(f"\n{df.tail(5)}")

## EMA High/Low Channel Strategy

**Your strategy**: A dynamic channel built from EMAs of the High and Low, filtered by a 200 EMA trend.

**Indicators**:
- `EMA_High(44)` — 44-period EMA applied to the High prices (upper channel)
- `EMA_Low(44)` — 44-period EMA applied to the Low prices (lower channel)
- `EMA_200` — 200-period EMA of Close (trend filter)

**Long Entry**: Close > EMA_200 AND Close > EMA_High(44) AND Close > EMA_Low(44)
*(Price is trending up AND has broken above the entire channel)*

**Exit Long**: Close < EMA_200 OR Close < EMA_Low(44)
*(Price breaks below trend or re-enters the channel from above)*

**Parameters to optimize**:
- `channel_period`: EMA period for H/L channel (20–80)
- `trend_period`: Trend filter EMA (100–300)

In [ ]:
# ============================================================
# Compute Indicators
# ============================================================
ema_high = talib.EMA(high, timeperiod=CHANNEL_PERIOD)
ema_low = talib.EMA(low, timeperiod=CHANNEL_PERIOD)
ema_trend = talib.EMA(close, timeperiod=TREND_PERIOD)

indicators = pd.DataFrame({
    'Close': close,
    'EMA_High_44': ema_high,
    'EMA_Low_44': ema_low,
    'EMA_200': ema_trend,
})
print("Indicators (last 10 bars):")
print(indicators.tail(10).to_string())

# Quick visual — last 200 bars (matching user's chart style)
fig, ax = plt.subplots(figsize=(16, 6))
n_show = min(200, len(close))
idx = slice(-n_show, None)

ax.plot(close.index[idx], close.values[idx], color='black', linewidth=1, label='Close')
ax.plot(close.index[idx], ema_high.values[idx], color='#00BFFF', linewidth=1.2, label=f'EMA High({CHANNEL_PERIOD})')
ax.plot(close.index[idx], ema_low.values[idx], color='#FF69B4', linewidth=1.2, label=f'EMA Low({CHANNEL_PERIOD})')
ax.plot(close.index[idx], ema_trend.values[idx], color='#2ECC71', linewidth=1.5, label=f'EMA({TREND_PERIOD})')

# Shade the channel
ax.fill_between(close.index[idx], ema_high.values[idx], ema_low.values[idx],
                color='#87CEEB', alpha=0.15, label='Channel')

ax.set_title(f'{TICKER} \u2014 EMA H/L Channel (last {n_show} bars)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylabel('Price')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Signal Logic
# ============================================================
def ema_hl_signals(close, high, low, channel_period, trend_period):
    """
    Generate entry/exit signals for EMA H/L Channel strategy.
    Long: Close > EMA(trend) AND Close > EMA_High(channel) AND Close > EMA_Low(channel)
    Exit: Close < EMA(trend) OR Close < EMA_Low(channel)
    """
    ema_h = pd.Series(talib.EMA(high, timeperiod=channel_period), index=close.index)
    ema_l = pd.Series(talib.EMA(low, timeperiod=channel_period), index=close.index)
    ema_t = pd.Series(talib.EMA(close, timeperiod=trend_period), index=close.index)
    
    # Bullish condition: above all three
    bullish = (close > ema_t) & (close > ema_h) & (close > ema_l)
    # Bearish condition: below trend OR below lower channel
    bearish = (close < ema_t) | (close < ema_l)
    
    # Entry on transition to bullish (1-bar delay for no lookahead)
    entries_raw = bullish & (~bullish.shift(1).fillna(False))
    exits_raw = bearish & (~bearish.shift(1).fillna(False))
    
    entries = entries_raw.shift(1).fillna(False)
    exits = exits_raw.shift(1).fillna(False)
    
    return entries, exits

# Test with default params
entries, exits = ema_hl_signals(close, high, low, CHANNEL_PERIOD, TREND_PERIOD)
print(f"Default params (Channel={CHANNEL_PERIOD}, Trend={TREND_PERIOD}):")
print(f"  Total entries: {entries.sum()}")
print(f"  Total exits:   {exits.sum()}")

In [ ]:
# ============================================================
# Parameter Ranges
# ============================================================
channel_range = list(range(20, 81, 4))    # 20 to 80 step 4
trend_range = list(range(100, 301, 20))   # 100 to 300 step 20

total_combos = len(channel_range) * len(trend_range)
print(f"Channel periods: {channel_range[0]}\u2013{channel_range[-1]} ({len(channel_range)} values)")
print(f"Trend periods: {trend_range[0]}\u2013{trend_range[-1]} ({len(trend_range)} values)")
print(f"Total combinations: {total_combos}")

In [ ]:
# ============================================================
# Grid Search (In-Sample)
# ============================================================
results = []

for i, ch in enumerate(channel_range):
    for tr in trend_range:
        try:
            entries, exits = ema_hl_signals(train_close, train_high, train_low, ch, tr)
            
            pf = vbt.Portfolio.from_signals(
                train_close, entries, exits,
                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
            )
            
            n_trades = pf.trades.count()
            if n_trades < 5:
                continue
            
            results.append({
                'channel': ch,
                'trend': tr,
                'is_sharpe': pf.sharpe_ratio(),
                'is_return': pf.total_return(),
                'is_max_dd': pf.max_drawdown(),
                'is_trades': n_trades,
                'is_win_rate': pf.trades.win_rate(),
                'is_profit_factor': pf.trades.profit_factor(),
            })
        except Exception:
            continue
    
    if (i + 1) % 5 == 0:
        print(f"\ud83d\udd04 [{(i+1)*len(trend_range)}/{total_combos} combos]")

results_df = pd.DataFrame(results).sort_values('is_sharpe', ascending=False)
print(f"\n\u2705 Completed: {len(results_df)} valid configs (min 5 trades)")
print(f"\nTop 10 by Sharpe:")
print(results_df.head(10).to_string(index=False))

# Check where user's default params rank
user_default = results_df[(results_df['channel'] == 44) & (results_df['trend'] == 200)]
if len(user_default) > 0:
    rank = (results_df['is_sharpe'] >= user_default.iloc[0]['is_sharpe']).sum()
    print(f"\n\ud83d\udccd Your default (44/200) ranks #{rank}/{len(results_df)} by IS Sharpe")
    print(f"   Sharpe={user_default.iloc[0]['is_sharpe']:.2f}, Return={user_default.iloc[0]['is_return']:.1%}, Trades={int(user_default.iloc[0]['is_trades'])}")

In [ ]:
# ============================================================
# Out-of-Sample Validation (Top 5)
# ============================================================
top5 = results_df.head(5)
oos_results = []
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for idx_i, (_, row) in enumerate(top5.iterrows()):
    ch, tr = int(row['channel']), int(row['trend'])
    
    entries, exits = ema_hl_signals(val_close, val_high, val_low, ch, tr)
    pf = vbt.Portfolio.from_signals(
        val_close, entries, exits,
        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
    )
    
    oos_sharpe = pf.sharpe_ratio()
    degradation = 1 - (oos_sharpe / row['is_sharpe']) if row['is_sharpe'] > 0 else np.nan
    
    oos_results.append({
        'channel': ch, 'trend': tr,
        'is_sharpe': row['is_sharpe'],
        'oos_sharpe': oos_sharpe,
        'degradation': degradation,
        'oos_return': pf.total_return(),
        'oos_max_dd': pf.max_drawdown(),
        'oos_trades': pf.trades.count(),
    })
    
    eq = pf.value()
    axes[idx_i].plot(eq.index, eq.values, color='#4A90D9')
    axes[idx_i].set_title(f"Ch={ch} Tr={tr}\nOOS Sharpe={oos_sharpe:.2f}", fontsize=9)
    axes[idx_i].tick_params(labelsize=7)
    if idx_i == 0:
        axes[idx_i].set_ylabel('Portfolio Value ($)')

plt.suptitle(f'{TICKER} \u2014 EMA H/L Channel OOS Validation (Top 5)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

oos_df = pd.DataFrame(oos_results)
print("\nIS vs OOS Comparison:")
print(oos_df.to_string(index=False))

# Also validate user's default 44/200
if len(user_default) > 0:
    entries_ud, exits_ud = ema_hl_signals(val_close, val_high, val_low, 44, 200)
    pf_ud = vbt.Portfolio.from_signals(val_close, entries_ud, exits_ud,
                                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
    print(f"\n\ud83d\udccd Your default (44/200) OOS: Sharpe={pf_ud.sharpe_ratio():.2f}, Return={pf_ud.total_return():.1%}, Trades={pf_ud.trades.count()}")

best = oos_df.sort_values('oos_sharpe', ascending=False).iloc[0]
BEST_CH = int(best['channel'])
BEST_TR = int(best['trend'])
print(f"\n\u2705 Best OOS config: Channel={BEST_CH}, Trend={BEST_TR}")
print(f"   IS Sharpe={best['is_sharpe']:.2f} \u2192 OOS Sharpe={best['oos_sharpe']:.2f} (degradation: {best['degradation']:.1%})")

In [ ]:
# ============================================================
# Parameter Sensitivity Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Channel period sensitivity (fix trend at best)
ch_sens = results_df[results_df['trend'] == BEST_TR].sort_values('channel')
if len(ch_sens) > 0:
    colors = ['#2ECC71' if s > 0 else '#E74C3C' for s in ch_sens['is_sharpe']]
    axes[0].bar(ch_sens['channel'], ch_sens['is_sharpe'], color=colors, alpha=0.8, width=3)
    axes[0].axvline(BEST_CH, color='#4A90D9', linestyle='--', linewidth=2, label=f'Best={BEST_CH}')
    if 44 in ch_sens['channel'].values:
        axes[0].axvline(44, color='orange', linestyle=':', linewidth=2, label='Your default=44')
    axes[0].set_xlabel('Channel Period')
    axes[0].set_ylabel('IS Sharpe Ratio')
    axes[0].set_title(f'Channel Sensitivity (Trend={BEST_TR})')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

# Trend period sensitivity (fix channel at best)
tr_sens = results_df[results_df['channel'] == BEST_CH].sort_values('trend')
if len(tr_sens) > 0:
    colors = ['#2ECC71' if s > 0 else '#E74C3C' for s in tr_sens['is_sharpe']]
    axes[1].bar(tr_sens['trend'], tr_sens['is_sharpe'], color=colors, alpha=0.8, width=15)
    axes[1].axvline(BEST_TR, color='#4A90D9', linestyle='--', linewidth=2, label=f'Best={BEST_TR}')
    if 200 in tr_sens['trend'].values:
        axes[1].axvline(200, color='orange', linestyle=':', linewidth=2, label='Your default=200')
    axes[1].set_xlabel('Trend Period')
    axes[1].set_ylabel('IS Sharpe Ratio')
    axes[1].set_title(f'Trend Sensitivity (Channel={BEST_CH})')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{TICKER} \u2014 EMA H/L Channel Parameter Sensitivity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Full-Sample Backtest (Best Parameters)
# ============================================================
entries, exits = ema_hl_signals(close, high, low, BEST_CH, BEST_TR)

pf = vbt.Portfolio.from_signals(
    close, entries, exits,
    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
)

total_ret = pf.total_return()
n_years = len(close) / 252
cagr = (1 + total_ret) ** (1/n_years) - 1
sharpe = pf.sharpe_ratio()
sortino = pf.sortino_ratio()
max_dd = pf.max_drawdown()
calmar = cagr / abs(max_dd) if max_dd != 0 else 0
n_trades = pf.trades.count()
win_rate = pf.trades.win_rate()
pf_ratio = pf.trades.profit_factor()

bh = vbt.Portfolio.from_holding(close, init_cash=INIT_CASH, freq='D')
bh_ret = bh.total_return()
bh_sharpe = bh.sharpe_ratio()

# Also run with user's default params for comparison
entries_44, exits_44 = ema_hl_signals(close, high, low, 44, 200)
pf_44 = vbt.Portfolio.from_signals(close, entries_44, exits_44,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')

print(f"{'='*60}")
print(f"EMA H/L CHANNEL \u2014 Full Sample Results")
print(f"{'='*60}")
print(f"Best Params: Channel={BEST_CH}, Trend={BEST_TR}")
print(f"Period: {close.index[0].strftime('%Y-%m-%d')} \u2192 {close.index[-1].strftime('%Y-%m-%d')}")
print(f"{'\u2500'*60}")
print(f"{'Metric':<20} {'Best Params':>12} {'Your 44/200':>12} {'Buy & Hold':>12}")
print(f"{'\u2500'*60}")
print(f"{'Total Return':<20} {total_ret:>12.1%} {pf_44.total_return():>12.1%} {bh_ret:>12.1%}")
cagr_44 = (1+pf_44.total_return())**(1/n_years)-1
print(f"{'CAGR':<20} {cagr:>12.1%} {cagr_44:>12.1%} {((1+bh_ret)**(1/n_years)-1):>12.1%}")
print(f"{'Sharpe':<20} {sharpe:>12.2f} {pf_44.sharpe_ratio():>12.2f} {bh_sharpe:>12.2f}")
print(f"{'Max Drawdown':<20} {max_dd:>12.1%} {pf_44.max_drawdown():>12.1%} {bh.max_drawdown():>12.1%}")
print(f"{'Trades':<20} {n_trades:>12d} {pf_44.trades.count():>12d} {'\u2014':>12}")
print(f"{'Win Rate':<20} {win_rate:>12.1%} {pf_44.trades.win_rate():>12.1%} {'\u2014':>12}")
print(f"{'Profit Factor':<20} {pf_ratio:>12.2f} {pf_44.trades.profit_factor():>12.2f} {'\u2014':>12}")

In [ ]:
# ============================================================
# Performance Dashboard
# ============================================================
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 1, height_ratios=[3, 1.2, 1], hspace=0.08)

STRAT_COLOR = '#4A90D9'
BENCH_COLOR = '#999999'
DEFAULT_COLOR = '#FFA500'
DD_COLOR = '#FF6B6B'

# --- Panel 1: Equity curve ---
ax1 = fig.add_subplot(gs[0])
eq = pf.value()
eq_44 = pf_44.value()
bh_eq = bh.value()

ax1.plot(eq.index, eq.values, color=STRAT_COLOR, linewidth=1.5,
         label=f'Best ({BEST_CH}/{BEST_TR})  Sharpe={sharpe:.2f}  Ret={total_ret:.1%}')
ax1.plot(eq_44.index, eq_44.values, color=DEFAULT_COLOR, linewidth=1.2, linestyle='-.',
         label=f'Your 44/200  Sharpe={pf_44.sharpe_ratio():.2f}  Ret={pf_44.total_return():.1%}')
ax1.plot(bh_eq.index, bh_eq.values, color=BENCH_COLOR, linewidth=1, linestyle='--',
         label=f'Buy & Hold  Sharpe={bh_sharpe:.2f}  Ret={bh_ret:.1%}')
ax1.set_ylabel('Portfolio Value ($)', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.tick_params(labelbottom=False)
ax1.set_title(
    f'{TICKER} \u2014 EMA H/L Channel Strategy\n'
    f'CAGR: {cagr:.1%}  Sharpe: {sharpe:.2f}  MaxDD: {max_dd:.1%}  Trades: {n_trades}',
    fontsize=13, fontweight='bold'
)

# Train/test split
split_date = close.index[split_idx]
ax1.axvline(split_date, color='gray', linestyle=':', linewidth=1, alpha=0.5)

# --- Panel 2: Price with channel overlay ---
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ema_h_best = talib.EMA(high, timeperiod=BEST_CH)
ema_l_best = talib.EMA(low, timeperiod=BEST_CH)
ema_t_best = talib.EMA(close, timeperiod=BEST_TR)

ax2.plot(close.index, close.values, color='black', linewidth=0.6, label='Close')
ax2.plot(close.index, ema_h_best, color='#00BFFF', linewidth=0.8, label=f'EMA High({BEST_CH})')
ax2.plot(close.index, ema_l_best, color='#FF69B4', linewidth=0.8, label=f'EMA Low({BEST_CH})')
ax2.plot(close.index, ema_t_best, color='#2ECC71', linewidth=1.0, label=f'EMA({BEST_TR})')
ax2.fill_between(close.index, ema_h_best, ema_l_best, color='#87CEEB', alpha=0.1)

# Buy/sell markers
buy_dates = entries[entries].index
sell_dates = exits[exits].index
ax2.scatter(buy_dates, close[buy_dates], marker='^', color='#2ECC71', s=20, zorder=5)
ax2.scatter(sell_dates, close[sell_dates], marker='v', color='#E74C3C', s=20, zorder=5)

ax2.set_ylabel('Price', fontsize=10)
ax2.legend(loc='upper left', fontsize=7, ncol=4)
ax2.grid(True, alpha=0.3)
ax2.tick_params(labelbottom=False)

# --- Panel 3: Drawdown ---
ax3 = fig.add_subplot(gs[2], sharex=ax1)
dd = pf.drawdown() * 100
ax3.fill_between(dd.index, dd.values, 0, color=DD_COLOR, alpha=0.4)
ax3.set_ylabel('Drawdown (%)', fontsize=10)
ax3.set_xlabel('Date', fontsize=11)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Trade Analysis
# ============================================================
trades = pf.trades.records_readable

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Trade returns distribution
trade_rets = trades['Return'].values * 100
winners = trade_rets[trade_rets > 0]
losers = trade_rets[trade_rets <= 0]
axes[0,0].hist(winners, bins=20, color='#2ECC71', alpha=0.7, label=f'Winners ({len(winners)})')
axes[0,0].hist(losers, bins=20, color='#E74C3C', alpha=0.7, label=f'Losers ({len(losers)})')
axes[0,0].set_xlabel('Return (%)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Trade Return Distribution')
axes[0,0].legend(fontsize=9)
axes[0,0].grid(True, alpha=0.3)

# 2. Cumulative P&L
cum_pnl = trades['PnL'].cumsum()
axes[0,1].plot(range(len(cum_pnl)), cum_pnl, color='#4A90D9', linewidth=1.5)
axes[0,1].fill_between(range(len(cum_pnl)), cum_pnl, 0,
                        where=cum_pnl >= 0, color='#2ECC71', alpha=0.2)
axes[0,1].fill_between(range(len(cum_pnl)), cum_pnl, 0,
                        where=cum_pnl < 0, color='#E74C3C', alpha=0.2)
axes[0,1].set_xlabel('Trade #')
axes[0,1].set_ylabel('Cumulative P&L ($)')
axes[0,1].set_title('Cumulative P&L by Trade')
axes[0,1].grid(True, alpha=0.3)

# 3. Monthly heatmap
port_rets = pf.returns()
monthly = port_rets.resample('ME').apply(lambda x: (1+x).prod() - 1)
monthly_pivot = pd.DataFrame({
    'Year': monthly.index.year,
    'Month': monthly.index.month,
    'Return': monthly.values
}).pivot_table(values='Return', index='Year', columns='Month', aggfunc='first')
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

im = axes[1,0].imshow(monthly_pivot.values * 100, cmap='RdYlGn', aspect='auto', vmin=-10, vmax=10)
axes[1,0].set_xticks(range(len(monthly_pivot.columns)))
axes[1,0].set_xticklabels(monthly_pivot.columns, fontsize=7)
axes[1,0].set_yticks(range(len(monthly_pivot.index)))
axes[1,0].set_yticklabels(monthly_pivot.index, fontsize=8)
for yi in range(len(monthly_pivot.index)):
    for xi in range(len(monthly_pivot.columns)):
        val = monthly_pivot.values[yi, xi]
        if not np.isnan(val):
            axes[1,0].text(xi, yi, f'{val*100:.1f}', ha='center', va='center', fontsize=7,
                          color='white' if abs(val) > 0.05 else 'black')
axes[1,0].set_title('Monthly Returns (%)')

# 4. Holding period
if 'Duration' in trades.columns:
    durations = pd.to_timedelta(trades['Duration']).dt.days
    axes[1,1].hist(durations, bins=20, color='#4A90D9', alpha=0.8)
    axes[1,1].set_xlabel('Holding Period (days)')
    axes[1,1].set_ylabel('Count')
    axes[1,1].set_title(f'Holding Period (avg: {durations.mean():.0f}d)')
    axes[1,1].grid(True, alpha=0.3)
else:
    axes[1,1].text(0.5, 0.5, 'Duration data\nnot available', ha='center', va='center')

plt.suptitle(f'{TICKER} \u2014 EMA H/L Channel Trade Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Monte Carlo FTMO Simulation
# ============================================================
N_SIMS = 10_000
trade_returns = trades['Return'].values

np.random.seed(42)
sim_equity = np.ones((N_SIMS, len(trade_returns) + 1)) * INIT_CASH
for i in range(N_SIMS):
    shuffled = np.random.choice(trade_returns, size=len(trade_returns), replace=True)
    sim_equity[i, 1:] = INIT_CASH * np.cumprod(1 + shuffled)

FTMO_TARGET = INIT_CASH * 1.10
FTMO_MAX_LOSS = INIT_CASH * 0.90
hit_target = (sim_equity.max(axis=1) >= FTMO_TARGET).mean() * 100
hit_loss = (sim_equity.min(axis=1) <= FTMO_MAX_LOSS).mean() * 100
pass_rate = ((sim_equity.max(axis=1) >= FTMO_TARGET) & (sim_equity.min(axis=1) > FTMO_MAX_LOSS)).mean() * 100

if pass_rate >= 60: verdict = "\u2705 PASS"
elif pass_rate >= 30: verdict = "\u26a0\ufe0f CONDITIONAL"
else: verdict = "\u274c FAIL"

fig, ax = plt.subplots(figsize=(14, 5))
for pct in [5, 25, 50, 75, 95]:
    line = np.percentile(sim_equity, pct, axis=0)
    alpha = 0.3 if pct in [5, 95] else 0.5 if pct in [25, 75] else 1.0
    lw = 0.8 if pct != 50 else 2.0
    style = '--' if pct != 50 else '-'
    ax.plot(range(len(line)), line, linewidth=lw, linestyle=style, alpha=alpha,
            color='#4A90D9', label=f'P{pct}')

ax.axhline(FTMO_TARGET, color='#2ECC71', linestyle='--', linewidth=1, label='Target (+10%)')
ax.axhline(FTMO_MAX_LOSS, color='#E74C3C', linestyle='--', linewidth=1, label='Max Loss (-10%)')
ax.fill_between(range(sim_equity.shape[1]),
                np.percentile(sim_equity, 5, axis=0),
                np.percentile(sim_equity, 95, axis=0),
                color='#4A90D9', alpha=0.1)

ax.set_xlabel('Trade #')
ax.set_ylabel('Equity ($)')
ax.set_title(f'{TICKER} \u2014 Monte Carlo FTMO ({N_SIMS:,} paths) | Pass Rate: {pass_rate:.1f}% | {verdict}',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"FTMO Results: Target hit {hit_target:.1f}% | Loss hit {hit_loss:.1f}% | Pass {pass_rate:.1f}% | {verdict}")

In [ ]:
# ============================================================
# Log to Google Sheets & Export
# ============================================================
try:
    sys.path.insert(0, os.path.join(os.getcwd(), 'lib'))
    from sheets_logger import SheetsLogger
    
    logger = SheetsLogger()
    logger.log_result(
        ticker=TICKER,
        strategy='EMA_HL_Channel',
        strategy_family='Trend_Following',
        fast_param=BEST_CH,
        slow_param=BEST_TR,
        filter_param=None,
        is_sharpe=float(best['is_sharpe']),
        oos_sharpe=float(best['oos_sharpe']),
        full_sharpe=sharpe,
        full_return=total_ret,
        full_max_dd=max_dd,
        full_trades=n_trades,
        win_rate=win_rate,
        profit_factor=pf_ratio,
        sharpe_degradation=float(best['degradation']),
        bh_sharpe=bh_sharpe,
        bh_return=bh_ret,
        mc_ftmo_pass_rate=pass_rate,
        mc_verdict=verdict,
        data_start=START_DATE,
        data_end=close.index[-1].strftime('%Y-%m-%d'),
        total_bars=len(close),
        notes=f"Channel={BEST_CH} Trend={BEST_TR} (user default: 44/200)"
    )
    print("\u2705 Results logged to Google Sheets")
except Exception as e:
    print(f"\u26a0\ufe0f Sheets logging skipped: {e}")

try:
    STRATEGY_NAME = "EMA_HL_Channel"
    PARAM_COLS = ["channel", "trend"]
    # Rename columns to match UNIVERSAL_EXPORT_CELL expectations
    results_df = results_df.rename(columns={
        'is_sharpe': 'sharpe_ratio',
        'is_return': 'total_return', 
        'is_max_dd': 'max_drawdown',
        'is_trades': 'total_trades',
        'is_win_rate': 'win_rate',
        'is_profit_factor': 'profit_factor',
    })

    exec(open(os.path.join('lib', 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())
except Exception as e:
    print(f"\u26a0\ufe0f Export skipped: {e}")